## Sequential training. 

This script takes the  the original annotation (for step 1) and the corrected additional labels and trains the models again. It can be re run multiple times by changig the "sequence" argument below. 

In [ ]:
sequence = 1
from opensoundscape import Audio, Spectrogram
from opensoundscape.annotations import BoxedAnnotations

import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt


In [ ]:
# for step 1 load the og annotation file and audio file
from Keys import annotation_file_tr_og, audio_file_file_tr_og
annotation_file = Path(annotation_file_tr_og)
audio_file = Path(audio_file_file_tr_og) 


For being able to use this labels in the opensoundscape training function we need to load them as object of the Boxedannotations class (they are not boxes but still)

In [ ]:
annot_list = [annotation_file]
audio_list = [audio_file]

# Create an object from Raven file
annotations_og = BoxedAnnotations.from_raven_files(
    raven_files=annot_list,
    audio_files=audio_list,
    annotation_column=None,  # Put None if you don't have an annotation column
)

# Inspect the object's .df attribute
# which contains the table of annotations
annotations_og.df.head()
annotations_og.df["annotation"] = "airgun"


# Parameters to use for label creation
clip_duration = 5
clip_overlap = 0
min_label_overlap = 0.25

# Create dataframe of one-hot labels
clip_labels = annotations_og.clip_labels(
    clip_duration = clip_duration,
    clip_overlap = clip_overlap,
    min_label_overlap = min_label_overlap,
)

Load the annotations from the inference step

In [ ]:
seq_annotations_folder=f"annotations_step{sequence}"
annotations_new = BoxedAnnotations.from_csv(Path(seq_annotations_folder).joinpath("ML_annotations_step.csv"))
annotations_new.df['annotation']='airgun'
clip_labels_new = annotations_new.clip_labels(
    clip_duration = clip_duration,
    clip_overlap = clip_overlap,
    min_label_overlap = min_label_overlap,
    audio_files=annotations_new.df['audio_file'].unique()
)


Join the previous step and the current step dataframe

In [ ]:
clip_labels_full = pd.concat([clip_labels, clip_labels_new])

In [ ]:
labels = clip_labels_full.sort_index(level="start_time")
n = len(labels)
print(n)
train_df = labels.iloc[: int(0.6 * n)]
val_df   = labels.iloc[int(0.6 * n) :]

for name, d in [("train", train_df), ("val", val_df),]:
    print(name, len(d), "clips,",d["airgun"].sum(), "with airgun")

429
train 257 clips, 140 with airgun
val 172 clips, 134 with airgun


Load the models and train the resnet and the efficien net

In [ ]:
# Create a CNN object designed to recognize 3-second samples
from opensoundscape import CNN

# Use resnet34 architecture
resnet = "resnet18"
effnet = "efficientnet_b0"

# Can use this code to get your classes, if needed
class_list = list(train_df.columns)
class_list
model1 = CNN(
    architecture=resnet,
    classes=class_list,
    sample_duration=clip_duration,  # 10s, selected above
    sample_rate=6000,
)

model2 = CNN(
    architecture=effnet,
    classes=class_list,
    sample_duration=clip_duration,  # 10s, selected above
    sample_rate=6000,
)


print(model1.device, model2.device)

cuda:0 cuda:0


Configure the wandb session

In [ ]:
import wandb
from Keys import api_key

# try:
wandb.login()
wandb_session = wandb.init(
    entity="AI_for_Ecology_ITAW",  # replace with your entity/group name
    project="Airgun_detector",  # replace with your project name
    name="First iteration resnet",
)
# except:  # if wandb.init fails, don't use wandb logging
#     print("failed to create wandb session. wandb session will be None")
#     wandb_session = None

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/biofelip/.netrc.
wandb: Currently logged in as: felipe-1092 (AI_for_Ecology_ITAW) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
checkpoint_folder = Path(seq_annotations_folder)

model1.train(train_df,
    val_df,
    steps=300,
    batch_size=16,
    log_interval=2,  # log progress every 100 batches
    validation_interval=2,
    # num_workers=num_workers,  # parallelized cpu tasks for preprocessing
    wandb_session=wandb_session,
    save_path=checkpoint_folder)  # location to save checkpoints)

For the next experiment close the previous session in wandb and add a new one with a new name. 

In [ ]:

wandb.finish()
wandb_session = wandb.init(
    entity="AI_for_Ecology_ITAW",  # replace with your entity/group name
    project="Airgun_detector",  # replace with your project name
    name="First iteration effnet",
)

Train the effnet. 

In [ ]:
effnet_folder = Path(checkpoint_folder).joinpath("effnet")
effnet_folder.mkdir(exist_ok=True)
model2.train(train_df,
    val_df,
    steps=300,
    batch_size=16,
    log_interval=2,  # log progress every 100 batches
    validation_interval=2,
    # num_workers=num_workers,  # parallelized cpu tasks for preprocessing
    wandb_session=wandb_session,
    save_path=effnet_folder)  # location to save checkpoints)

Load the feature extractor. 

In [ ]:
import bioacoustics_model_zoo as bmz
perch = bmz.Perch2ONNX(headless=True)
train_emb=np.asanyarray(perch.embed(train_df, return_dfs=False, return_preds=False))
val_emb=perch.embed(val_df, return_dfs=False, return_preds=False)

Train the logistic regression. The AUC is reduced from  the first attempt but not excesively. 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
# 3. Small classifier on top
clf = LogisticRegression(max_iter=5000, class_weight="balanced")   # balanced: you have few negatives
clf.fit(train_emb, train_df['airgun'])

val_scores = clf.predict_proba(val_emb)[:, 1]
print("val AUC:", roc_auc_score(val_df["airgun"], val_scores))


val AUC: 0.856637863315004


save the shallow classifier

In [ ]:
import joblib
joblib.dump(clf, Path(checkpoint_folder).joinpath("Shallow_classifier.joblib"))   # save

['annotations_step1/effnet/Shallow_classifier.joblib']

Now we can take this models for the step numbet 2 and repeat. 